# 22 — Extract the visually reviewed YouTube long-workout batch v2

This notebook downloads the two new long videos at **720p**, verifies the
first-pass visual timelines committed by Codex, and extracts continuous pose
landmarks. The combined source duration is **61:59**.

Use a **CPU runtime**. MediaPipe runs on the CPU here; a T4 does not provide
enough benefit to justify GPU units.

The visual first pass is already complete: 118 intervals separate exercise,
rest, preview, equipment change, and transition motion. Unsupported and
compound movements remain `unknown`. The notebook will:

1. download the videos temporarily at 720p H.264;
2. download and checksum the committed first-pass timelines;
3. create compact QA contact sheets and run the structural audit;
4. extract full-video 17-joint landmarks at 15 FPS;
5. download `calorie_youtube_long_workouts_v2_review_landmarks.zip`.

All rows remain `use_for_training=false` and `needs_second_review=true`.
Independent review and adjudication are still required before training clips
may be created. Raw videos remain temporary and are excluded from the ZIP.


In [ ]:
%pip install -q --upgrade yt-dlp mediapipe==0.10.35 opencv-python-headless pandas==2.2.3 tqdm


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from IPython.display import Image, clear_output, display
from google.colab import files
from tqdm.auto import tqdm

TARGET_FPS = 15.0
PINNED_DATA_COMMIT = 'f395f4fb1b90d8894555cf28be2c9490d0124a74'
RAW_BASE = f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_DATA_COMMIT}'
WORK_ROOT = Path('/content/calorie_youtube_long_v2_work')
VIDEO_ROOT = WORK_ROOT / 'videos'
LABEL_ROOT = WORK_ROOT / 'first_pass_labels'
CONTACT_ROOT = WORK_ROOT / 'contact_sheets'
OUTPUT_ROOT = Path('/content/calorie_youtube_long_workouts_v2_review_landmarks')
LANDMARK_ROOT = OUTPUT_ROOT / 'continuous_landmarks'
for directory in (VIDEO_ROOT, LABEL_ROOT, CONTACT_ROOT, OUTPUT_ROOT, LANDMARK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {'video_id': 'zxqCV_eQHZY', 'creator': 'Legally Swole', 'stem': 'legally_swole__zxqCV_eQHZY', 'duration_s': 1898},
    {'video_id': 'Fk1wnNX4wQ8', 'creator': 'Daniel Nikolic', 'stem': 'daniel_nikolic__Fk1wnNX4wQ8', 'duration_s': 1821},
]
EXPECTED_SHA256 = {
    'candidates.csv': 'fb8af2597dfb93942bb2512e524352036ac6ea5e6ea2df37c1ddd29c0c78da34',
    'daniel_nikolic__Fk1wnNX4wQ8.labels.csv': '1fe790126350b86d1e56ae42f0007fbd78251ffed2e90d15d7cd9796a0455dc4',
    'legally_swole__zxqCV_eQHZY.labels.csv': '24cdc891d8f00a3bbdb8ad5c026985c374c2b1c8c15d0b5047685c23426b35c5',
    'audit_timeline_annotations.py': '2b696a0461bfba7e877fc5f1c1aef5c6e4cbc45d27ed6c2a8dbaabadafd245ef',
    'classes.json': '31bf6227d67528fc4f3fc6ecd449dfba07493dd7d1f15c57217e799837ec3207',
}
print('Runtime: CPU')
print('Pinned visual-label commit:', PINNED_DATA_COMMIT)
print('Final output:', OUTPUT_ROOT.with_suffix('.zip'))


## Download temporary 720p videos and pinned first-pass labels

The 720p H.264 videos preserve small-joint visibility for pose extraction
without the unnecessary cost of 1080p processing. They are deleted with the
Colab runtime and never enter the output archive. Every repository input is
pinned to the visual-review commit and verified by SHA-256.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def pinned_download(relative_url, destination, expected_sha):
    destination = Path(destination)
    if not destination.exists() or sha256(destination) != expected_sha:
        temporary = destination.with_suffix(destination.suffix + '.part')
        temporary.unlink(missing_ok=True)
        urllib.request.urlretrieve(f'{RAW_BASE}/{relative_url}', temporary)
        assert sha256(temporary) == expected_sha, f'Checksum mismatch: {destination.name}'
        temporary.replace(destination)
    assert sha256(destination) == expected_sha
    return destination


candidates_path = pinned_download(
    'ml/data/youtube_long_workouts_v2/candidates.csv',
    WORK_ROOT / 'candidates.csv', EXPECTED_SHA256['candidates.csv']
)
for source in SOURCES:
    label_name = f"{source['stem']}.labels.csv"
    pinned_download(
        f'ml/data/youtube_long_workouts_v2/annotations/first_pass/{label_name}',
        LABEL_ROOT / label_name, EXPECTED_SHA256[label_name]
    )

    video_template = VIDEO_ROOT / f"{source['stem']}.%(ext)s"
    matches = list(VIDEO_ROOT.glob(f"{source['stem']}.*"))
    if not matches:
        subprocess.run([
            'yt-dlp', '--no-playlist',
            '-f', '136/bestvideo[height=720][ext=mp4]/bestvideo[height<=720][ext=mp4]',
            '--merge-output-format', 'mp4', '-o', str(video_template),
            f"https://www.youtube.com/watch?v={source['video_id']}",
        ], check=True)
        matches = list(VIDEO_ROOT.glob(f"{source['stem']}.*"))
    assert len(matches) == 1, (source['stem'], matches)
    source['video_path'] = matches[0]
    cap = cv2.VideoCapture(str(matches[0]))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    cap.release()
    assert height == 720, f"Expected 720p source for {source['video_id']}, got {width}x{height}"
    print(source['video_id'], matches[0].name, f'{width}x{height}', f'{fps:.2f} FPS', frame_count, 'frames')


## Build compact QA contact sheets

The committed timelines came from a four-second visual pass. These compact
start/middle/end sheets are reproducibility evidence and a quick landmark-QA
aid; they do not replace the later independent second review.


In [ ]:
def read_frame(video_path, time_s):
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, time_s) * 1000.0)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Could not decode {video_path} at {time_s:.2f}s')
    return frame


def make_contact_sheet(video_path, row, output_path):
    start, end = float(row.start_s), float(row.end_s)
    margin = min(1.0, max(0.05, (end - start) * 0.05))
    times = [start + margin, (start + end) / 2.0, max(start + margin, end - margin)]
    panels = []
    for time_s in times:
        frame = read_frame(video_path, time_s)
        height, width = frame.shape[:2]
        resized = cv2.resize(frame, (480, max(1, round(height * 480 / width))))
        cv2.putText(resized, f'{time_s:.1f}s', (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        panels.append(resized)
    cv2.imwrite(str(output_path), np.concatenate(panels, axis=1))


label_tables = {}
contact_index = []
for source in SOURCES:
    label_path = LABEL_ROOT / f"{source['stem']}.labels.csv"
    table = pd.read_csv(label_path).fillna('')
    label_tables[source['video_id']] = table
    source_contacts = CONTACT_ROOT / source['video_id']
    source_contacts.mkdir(parents=True, exist_ok=True)
    for index, row in tqdm(list(table.iterrows()), desc=f"QA sheets {source['video_id']}"):
        output_path = source_contacts / f'{index:03d}.jpg'
        if not output_path.exists():
            make_contact_sheet(source['video_path'], row, output_path)
        contact_index.append({'video_id': source['video_id'], 'row_index': index, 'path': str(output_path)})
pd.DataFrame(contact_index).to_csv(WORK_ROOT / 'contact_sheet_index.csv', index=False)
print('First-pass rows / QA sheets:', len(contact_index))


## Inspect the committed first-pass summary

Supported labels appear only where the visible motion matches the current
ontology. Compound, partial-range, and unsupported exercises remain
`unknown`. No row is enabled for training.


In [ ]:
summary_rows = []
for source in SOURCES:
    table = label_tables[source['video_id']]
    assert (table.review_status == 'reviewed').all()
    assert (table.reviewer == 'codex_visual_review_2026-10-15').all()
    assert (table.use_for_training.astype(str).str.lower() == 'false').all()
    assert (table.needs_second_review.astype(str).str.lower() == 'true').all()
    exercise = table[table.segment_type == 'exercise']
    summary_rows.append({
        'video_id': source['video_id'],
        'timeline_rows': len(table),
        'exercise_rows': len(exercise),
        'supported_rows': int((exercise.current_tcn_label != 'unknown').sum()),
        'unknown_exercise_rows': int((exercise.current_tcn_label == 'unknown').sum()),
        'duration_s': float(table.end_s.max()),
    })
display(pd.DataFrame(summary_rows))
display(pd.concat(label_tables.values()).query("current_tcn_label != 'unknown'")
        [['video_id', 'start_time', 'end_time', 'fine_label', 'current_tcn_label', 'annotation_confidence']])


## Audit the complete first-pass timelines

The audit must report zero structural errors. Four-second boundary warnings
are expected and are preserved honestly for the independent reviewer.


In [ ]:
AUDIT_SCRIPT = pinned_download(
    'ml/scripts/audit_timeline_annotations.py',
    WORK_ROOT / 'audit_timeline_annotations.py', EXPECTED_SHA256['audit_timeline_annotations.py']
)
CLASSES_PATH = pinned_download(
    'ml/config/classes.json',
    WORK_ROOT / 'classes.json', EXPECTED_SHA256['classes.json']
)

audit_path = OUTPUT_ROOT / 'first_pass_annotation_audit.json'
subprocess.run([
    'python3', str(AUDIT_SCRIPT), str(LABEL_ROOT),
    '--classes', str(CLASSES_PATH), '--report', str(audit_path),
], check=True)
audit = json.loads(audit_path.read_text())
assert audit['files'] == 2
assert audit['rows'] == 118
assert audit['errors'] == 0
print('First-pass audit passed:', audit['rows'], 'rows;', audit['warnings'], 'honest 4-second precision warnings')


## Extract continuous landmarks at 15 FPS

The arrays cover each complete source video and retain timestamps plus a per-frame detection mask. They are intentionally not split into labelled training clips in this notebook.

In [ ]:
MODEL_PATH = WORK_ROOT / 'pose_landmarker_full.task'
MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
    'pose_landmarker_full/float16/latest/pose_landmarker_full.task'
)
if not MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print('Pose model MB:', round(MODEL_PATH.stat().st_size / 1e6, 2))

KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    scale = max(
        float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])),
        float(np.linalg.norm(shoulder_center - hip_center)), 1e-6,
    )
    return np.concatenate([(xyz - hip_center) / scale, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    shape = sequence.shape
    flat = sequence.reshape(shape[0], -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction='both').fillna(0).to_numpy()
    return filled.reshape(shape).astype(np.float32)


def extract_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open {video_path}')
    source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    next_sample, frame_index = 0.0, 0
    poses, timestamps, detected = [], [], []
    options = mp.tasks.vision.PoseLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH)),
        running_mode=mp.tasks.vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5, min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5, output_segmentation_masks=False,
    )
    progress = tqdm(total=source_frames, desc=Path(video_path).stem, unit='frames')
    with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            time_s = frame_index / source_fps
            frame_index += 1
            progress.update(1)
            if time_s + 1e-9 < next_sample:
                continue
            next_sample += 1.0 / TARGET_FPS
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect_for_video(image, int(round(time_s * 1000)))
            if result.pose_landmarks:
                points = result.pose_landmarks[0]
                pose = np.asarray([[points[i].x, points[i].y, points[i].z, float(points[i].visibility or 0.0)] for i in KEEP_17], dtype=np.float32)
                poses.append(normalize_pose(pose))
                detected.append(True)
            else:
                poses.append(np.full((17, 4), np.nan, dtype=np.float32))
                detected.append(False)
            timestamps.append(time_s)
    progress.close()
    cap.release()
    if not poses:
        raise RuntimeError(f'No frames decoded from {video_path}')
    return fill_missing(np.stack(poses)), np.asarray(timestamps, dtype=np.float32), np.asarray(detected, dtype=bool), source_fps


In [ ]:
quality_rows = []
for source in SOURCES:
    output_path = LANDMARK_ROOT / f"{source['stem']}.npz"
    if output_path.exists():
        item = np.load(output_path)
        landmarks, timestamps, detected = item['landmarks'], item['timestamps_s'], item['detected']
        source_fps = float(item['source_fps'])
    else:
        landmarks, timestamps, detected, source_fps = extract_video(source['video_path'])
        np.savez_compressed(
            output_path, landmarks=landmarks.astype(np.float32),
            timestamps_s=timestamps.astype(np.float32), detected=detected,
            source_fps=np.float32(source_fps), target_fps=np.float32(TARGET_FPS),
            video_id=source['video_id'], group_id=f"youtube:{source['video_id']}",
        )
    detection_rate = float(detected.mean())
    quality_rows.append({
        'video_id': source['video_id'], 'creator': source['creator'],
        'group_id': f"youtube:{source['video_id']}",
        'landmark_path': str(output_path.relative_to(OUTPUT_ROOT)),
        'sampled_frames': len(landmarks), 'duration_s': float(timestamps[-1]),
        'target_fps': TARGET_FPS, 'source_fps': source_fps,
        'detection_rate': detection_rate,
        'status': 'ok' if detection_rate >= 0.80 else 'needs_pose_review',
        'label_status': 'first_pass_reviewed_needs_independent_second_review',
        'use_for_training': False,
    })
    print(source['video_id'], landmarks.shape, f'detection={detection_rate:.3f}')
quality = pd.DataFrame(quality_rows)
quality.to_csv(OUTPUT_ROOT / 'continuous_landmark_manifest.csv', index=False)
display(quality)
assert len(quality) == 2
assert quality.sampled_frames.sum() > 50000, 'Unexpectedly few landmark frames'


## Package the review evidence and continuous landmarks

The archive includes the two first-pass CSVs, structural audit, continuous
landmark arrays, compact QA sheets, manifest, metadata, and checksums. It
excludes raw videos. Return this ZIP to the chat for pose-quality inspection
and independent label review.


In [ ]:
shutil.copytree(LABEL_ROOT, OUTPUT_ROOT / 'first_pass_labels', dirs_exist_ok=True)
shutil.copytree(CONTACT_ROOT, OUTPUT_ROOT / 'contact_sheets', dirs_exist_ok=True)
shutil.copy2(candidates_path, OUTPUT_ROOT / 'candidates.csv')
run_metadata = {
    'notebook': '22_colab_review_extract_youtube_long_v2.ipynb',
    'runtime': 'CPU', 'target_fps': TARGET_FPS,
    'pinned_data_commit': PINNED_DATA_COMMIT,
    'training_allowed': False,
    'visual_review': 'four-second contact-sheet pass committed at f395f4fb1b90d8894555cf28be2c9490d0124a74',
    'next_step': 'pose-quality inspection plus independent second review and adjudication before training',
}
(OUTPUT_ROOT / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')

checksum_rows = []
for path in sorted(p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.name != 'SHA256SUMS.txt'):
    checksum_rows.append(f"{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}")
(OUTPUT_ROOT / 'SHA256SUMS.txt').write_text('\n'.join(checksum_rows) + '\n')

assert not list(OUTPUT_ROOT.rglob('*.mp4')), 'Raw video must not enter the output archive'
assert len(list(LANDMARK_ROOT.glob('*.npz'))) == 2
zip_path = Path(shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT))
print('ZIP:', zip_path)
print('ZIP size MB:', round(zip_path.stat().st_size / 1e6, 2))
print('SHA-256:', sha256(zip_path))
files.download(str(zip_path))
